# PDF → Structured JSON Export

Exportiert jedes PDF als ein JSON mit drei Top-Level-Keys:

| Feld | Inhalt |
|---|---|
| **metainfo** | SHA1, Seitenzahl, Block-/Tabellen-/Fußnoten-Counts, Company-Name |
| **content** | Seitenweise geordnet, Reading-Order via `body.children`, lückenlos durchnummeriert |
| **tables** | Jede Tabelle in 3 Formaten: Markdown, HTML, JSON-Grid |

## Content-Items
- Text-Blöcke: `text`, `type`, `text_id`, optional `orig`, `level`, `enumerated`, `marker`
- Gruppen-Kontext: optional `group_id`, `group_name`, `group_label`
- Tabellen-Referenzen: `table_id`
- Jede Seite hat `page_dimensions` (PDF-Punkte)

## Docling-Settings
- **OCR**: EasyOCR (en), `force_full_page_ocr=False`
- **Table Structure**: ACCURATE + `do_cell_matching=True`
- **Backend**: `PyPdfiumDocumentBackend`
- **Pictures**: deaktiviert (`generate_picture_images=False`)

In [1]:
import json
import csv
import re
import hashlib
import datetime
from pathlib import Path
from typing import List, Dict, Optional, Tuple, Generator

from tabulate import tabulate

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat, ConversionStatus
from docling.datamodel.pipeline_options import (
    ThreadedPdfPipelineOptions,
    TableFormerMode,
    EasyOcrOptions,
)
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.settings import settings

settings.perf.page_batch_size = 64

In [2]:
GLYPH_PATTERNS = [
    (re.compile(r'/zero\.tnum'),  '0'),
    (re.compile(r'/one\.tnum'),   '1'),
    (re.compile(r'/two\.tnum'),   '2'),
    (re.compile(r'/three\.tnum'), '3'),
    (re.compile(r'/four\.tnum'),  '4'),
    (re.compile(r'/five\.tnum'),  '5'),
    (re.compile(r'/six\.tnum'),   '6'),
    (re.compile(r'/seven\.tnum'), '7'),
    (re.compile(r'/eight\.tnum'), '8'),
    (re.compile(r'/nine\.tnum'),  '9'),
    (re.compile(r'/dollar\.pl'),  '$'),
    (re.compile(r'/percent'),      '%'),
    (re.compile(r'/([A-Z])\.cap'), lambda m: m.group(1)),
    (re.compile(r'/([a-z])\.lc'),  lambda m: m.group(1)),
    (re.compile(r'glyph<[^>]*>'),  ''),
    (re.compile(r'/[a-z]+\.(lf|sc|smcp|onum|pnum|subs|sups|tnum|pl)'), ''),
    (re.compile(r' *\.{2,}'), ''),   # dot-leader fill characters
    (re.compile('\ufffd'), ''),       # unicode replacement character
]


def _clean_cell_text(text: str) -> str:
    """Apply glyph cleanup patterns to text."""
    for pattern, repl in GLYPH_PATTERNS:
        text = pattern.sub(repl, text)
    return text.strip()


LABEL_MAP = {"text": "paragraph"}


class StructuredExporter:
    """PDF -> structured JSON (metainfo, content[], tables[])."""

    def __init__(self, source_dir: str, csv_path: str = None, test_mode: bool = True):
        self.source_dir = Path(source_dir)
        self.test_mode = test_mode

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        self.output_dir = self.source_dir.parent / "docling_output" / timestamp
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.MAX_PDFS = 5 if test_mode else None

        # Company-Name lookup (optional CSV: ticker -> company)
        self.company_lookup = {}
        if csv_path:
            self._load_csv(csv_path)

        # GPU
        accelerator_options = AcceleratorOptions(
            device=AcceleratorDevice.CUDA,
            num_threads=8,
        )

        # OCR
        ocr_options = EasyOcrOptions(lang=["en"], force_full_page_ocr=False, use_gpu=True)

        # Pipeline
        pipeline_options = ThreadedPdfPipelineOptions(
            do_ocr=True,
            ocr_options=ocr_options,
            do_table_structure=True,
            accelerator_options=accelerator_options,
            layout_batch_size=64,
            table_batch_size=8,
        )
        pipeline_options.table_structure_options.mode = TableFormerMode.ACCURATE
        pipeline_options.table_structure_options.do_cell_matching = True
        pipeline_options.generate_picture_images = False

        self.converter = DocumentConverter(
            format_options={
                InputFormat.PDF: PdfFormatOption(
                    pipeline_options=pipeline_options,
                    backend=PyPdfiumDocumentBackend,
                )
            }
        )

        print(f"Output: {self.output_dir}")
        if test_mode:
            print(f"Test-Mode: First {self.MAX_PDFS} PDFs")

    def _load_csv(self, csv_path: str):
        """Load ticker->company mapping from CSV."""
        path = Path(csv_path)
        if not path.exists():
            print(f"CSV not found: {csv_path}")
            return
        with open(path, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                ticker = row.get("ticker", row.get("Ticker", ""))
                company = row.get("company_name", row.get("Company", row.get("company", "")))
                if ticker:
                    self.company_lookup[ticker.upper()] = company
        print(f"Loaded {len(self.company_lookup)} companies from CSV")

    # -- Children Traversal -----------------------------------------------

    def _resolve_children(self, doc, children, group_info=None) -> Generator:
        """Recursively resolve refs. Flatten groups, yield (leaf, group_info)."""
        for ref_item in children:
            item = ref_item.resolve(doc)
            ref_path = item.self_ref if hasattr(item, "self_ref") else ""
            if ref_path.startswith("#/groups/"):
                parts = ref_path.split("/")
                gid = int(parts[2]) if len(parts) == 3 else None
                new_group = {
                    "group_id": gid,
                    "group_name": getattr(item, "name", None),
                    "group_label": str(item.label) if hasattr(item, "label") else None,
                }
                if hasattr(item, "children") and item.children:
                    yield from self._resolve_children(doc, item.children, new_group)
            else:
                yield item, group_info

    @staticmethod
    def _get_ref_id(item) -> Tuple[Optional[str], Optional[int]]:
        ref = item.self_ref if hasattr(item, "self_ref") else ""
        parts = ref.split("/")
        if len(parts) == 3:
            return parts[1], int(parts[2])
        return None, None

    @staticmethod
    def _get_page(item) -> Optional[int]:
        if hasattr(item, "prov") and item.prov:
            return item.prov[0].page_no
        return None

    @staticmethod
    def _map_label(label: str) -> str:
        return LABEL_MAP.get(label, label)

    # -- Content Assembly --------------------------------------------------

    def _build_text_block(self, item, idx, group_info) -> Dict:
        label = str(item.label) if hasattr(item, "label") else "paragraph"
        raw_text = item.text if hasattr(item, "text") and item.text else ""
        clean_text = raw_text.strip()

        block = {
            "text": clean_text,
            "type": self._map_label(label),
            "text_id": idx,
        }

        if raw_text and raw_text != clean_text:
            block["orig"] = raw_text

        if hasattr(item, "level") and item.level is not None:
            block["level"] = item.level

        if hasattr(item, "enumerated") and item.enumerated is not None:
            block["enumerated"] = item.enumerated
        if hasattr(item, "marker") and item.marker:
            block["marker"] = item.marker

        if group_info:
            block["group_id"] = group_info["group_id"]
            if group_info["group_name"]:
                block["group_name"] = group_info["group_name"]
            if group_info["group_label"]:
                block["group_label"] = group_info["group_label"]

        return block

    def assemble_content(self, doc) -> List[Dict]:
        """Build page-grouped content following body.children reading order."""
        page_blocks = {}

        # Body items (main content, reading order)
        for item, group_info in self._resolve_children(doc, doc.body.children):
            collection, idx = self._get_ref_id(item)
            page = self._get_page(item)
            if not page:
                continue

            if collection == "texts":
                block = self._build_text_block(item, idx, group_info)
                page_blocks.setdefault(page, []).append(block)

            elif collection == "tables":
                block = {"type": "table", "table_id": idx}
                page_blocks.setdefault(page, []).append(block)

        # Furniture (page headers/footers) — deprecated in newer Docling
        try:
            if hasattr(doc, "furniture") and doc.furniture and doc.furniture.children:
                for item, group_info in self._resolve_children(doc, doc.furniture.children):
                    collection, idx = self._get_ref_id(item)
                    page = self._get_page(item)
                    if not page or collection != "texts":
                        continue
                    block = self._build_text_block(item, idx, group_info)
                    blocks = page_blocks.setdefault(page, [])
                    if "header" in block["type"]:
                        blocks.insert(0, block)
                    else:
                        blocks.append(block)
        except Exception:
            pass

        # Build result: gap-free page sequence with page_dimensions
        max_page = max(doc.pages.keys()) if doc.pages else (
            max(page_blocks.keys()) if page_blocks else 0
        )

        result = []
        for p in range(1, max_page + 1):
            entry = {
                "page": p,
                "content": page_blocks.get(p, []),
            }
            page_obj = doc.pages.get(p)
            if page_obj and hasattr(page_obj, "size") and page_obj.size:
                entry["page_dimensions"] = {
                    "l": 0,
                    "t": 0,
                    "r": page_obj.size.width,
                    "b": page_obj.size.height,
                }
            result.append(entry)

        return result

    # -- Tables ------------------------------------------------------------

    @staticmethod
    def _table_grid_json(table) -> Dict:
        if not (hasattr(table, "data") and table.data):
            return {"num_rows": 0, "num_cols": 0, "cells": []}
        grid = table.data
        cells = []
        for cell in grid.table_cells:
            cells.append({
                "text": _clean_cell_text(cell.text),
                "row": cell.start_row_offset_idx,
                "col": cell.start_col_offset_idx,
                "row_span": cell.end_row_offset_idx - cell.start_row_offset_idx,
                "col_span": cell.end_col_offset_idx - cell.start_col_offset_idx,
                "is_col_header": cell.column_header,
                "is_row_header": cell.row_header,
            })
        return {"num_rows": grid.num_rows, "num_cols": grid.num_cols, "cells": cells}

    @staticmethod
    def _render_table_markdown(table) -> str:
        """GFM markdown without colspan/rowspan duplication."""
        if not (hasattr(table, "data") and table.data):
            return ""
        data = table.data
        if data.num_rows == 0 or data.num_cols == 0:
            return ""

        grid = [[""] * data.num_cols for _ in range(data.num_rows)]
        for cell in data.table_cells:
            text = _clean_cell_text(cell.text or "").replace("\n", " ").replace("|", "&#124;")
            r, c = cell.start_row_offset_idx, cell.start_col_offset_idx
            if r < data.num_rows and c < data.num_cols:
                grid[r][c] = text

        try:
            return tabulate(grid[1:], headers=grid[0], tablefmt="github")
        except ValueError:
            return tabulate(grid[1:], headers=grid[0], tablefmt="github",
                            disable_numparse=True)

    def assemble_tables(self, doc) -> List[Dict]:
        tables = []
        if not (hasattr(doc, "tables") and doc.tables):
            return tables

        for idx, table in enumerate(doc.tables):
            prov = table.prov[0] if (hasattr(table, "prov") and table.prov) else None

            md = self._render_table_markdown(table)

            try:
                html = table.export_to_html(doc)
            except TypeError:
                html = table.export_to_html()
            html = _clean_cell_text(html)

            tables.append({
                "table_id": idx,
                "page": prov.page_no if prov else None,
                "bbox": [prov.bbox.l, prov.bbox.t, prov.bbox.r, prov.bbox.b] if prov else None,
                "#-rows": table.data.num_rows if (hasattr(table, "data") and table.data) else 0,
                "#-cols": table.data.num_cols if (hasattr(table, "data") and table.data) else 0,
                "markdown": md,
                "html": html,
                "json": self._table_grid_json(table),
            })

        return tables

    # -- Metainfo ----------------------------------------------------------

    def build_metainfo(self, pdf_path: Path, doc) -> Dict:
        with open(pdf_path, "rb") as f:
            sha1 = hashlib.sha1(f.read()).hexdigest()

        num_text = 0
        num_footnotes = 0
        num_equations = 0
        if hasattr(doc, "texts") and doc.texts:
            num_text = len(doc.texts)
            for t in doc.texts:
                label = str(t.label) if hasattr(t, "label") else ""
                if label == "footnote":
                    num_footnotes += 1
                elif label == "formula":
                    num_equations += 1

        ticker = pdf_path.stem.split("_")[0].upper()
        company_name = self.company_lookup.get(ticker, ticker)

        return {
            "sha1_name": sha1,
            "doc_id": pdf_path.stem,
            "pages_amount": len(doc.pages) if doc.pages else 0,
            "text_blocks_amount": num_text,
            "tables_amount": len(doc.tables) if (hasattr(doc, "tables") and doc.tables) else 0,
            "equations_amount": num_equations,
            "footnotes_amount": num_footnotes,
            "company_name": company_name,
        }

    # -- Process -----------------------------------------------------------

    def process_doc(self, pdf_path: Path, result) -> Optional[Dict]:
        pdf_name = pdf_path.stem
        print(f"\n{pdf_name}")

        try:
            doc = result.document

            content = self.assemble_content(doc)
            tables = self.assemble_tables(doc)
            metainfo = self.build_metainfo(pdf_path, doc)

            output = {
                "metainfo": metainfo,
                "content": content,
                "tables": tables,
            }

            out_file = self.output_dir / f"{pdf_name}.json"
            with open(out_file, "w", encoding="utf-8") as f:
                json.dump(output, f, indent=2, ensure_ascii=False)

            # Save native Docling JSON (for HybridChunker in chunking.ipynb)
            native_dir = self.output_dir / "native"
            native_dir.mkdir(exist_ok=True)
            with open(native_dir / f"{pdf_name}.json", "w", encoding="utf-8") as f:
                f.write(doc.model_dump_json(indent=2))

            num_blocks = sum(len(p["content"]) for p in content)
            print(f"  {metainfo['pages_amount']} pages | {len(tables)} tables | {num_blocks} blocks | {metainfo['footnotes_amount']} fn")
            print(f"  -> {out_file.name}")
            return output

        except Exception as e:
            print(f"  Error: {e}")
            import traceback
            traceback.print_exc()
            return None

    def process_all(self) -> Dict:
        pdf_files = sorted(self.source_dir.glob("*.pdf"))
        if self.test_mode and self.MAX_PDFS:
            pdf_files = pdf_files[:self.MAX_PDFS]

        print(f"{'TEST: ' if self.test_mode else ''}{len(pdf_files)} PDFs\n")
        print("Batch-Conversion...")

        stats = {"success": 0, "error": 0}

        results = self.converter.convert_all(
            [str(p) for p in pdf_files],
            raises_on_error=False,
        )

        for result in results:
            pdf_path = Path(result.input.file)
            if not pdf_path.is_absolute():
                pdf_path = self.source_dir / pdf_path.name

            if result.status != ConversionStatus.SUCCESS:
                print(f"\n{pdf_path.stem} skipped ({result.status})")
                stats["error"] += 1
                continue

            if self.process_doc(pdf_path, result):
                stats["success"] += 1
            else:
                stats["error"] += 1

        print(f"\n{'='*40}")
        print(f"OK: {stats['success']} | Errors: {stats['error']}")
        return stats

In [3]:
exporter = StructuredExporter(
    source_dir="C:/Users/phili/PycharmProjects/doc/data/src/fin_test_set",
    csv_path=None,  # Optional: Pfad zu subset.csv mit Ticker -> Company-Name
    test_mode=False,
)
exporter.process_all()

Output: C:\Users\phili\PycharmProjects\doc\data\src\docling_output\20260409_221927
27 PDFs

Batch-Conversion...


C:\Users\phili\PycharmProjects\doc\.venv\Lib\site-packages\docling\models\stages\ocr\easyocr_model.py:68: UserWarning: Deprecated field. Better to set the `accelerator_options.device` in `pipeline_options`. When `use_gpu and accelerator_options.device == AcceleratorDevice.CUDA` the GPU is used to run EasyOCR. Otherwise, EasyOCR runs in CPU.
  warnings.warn(


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]


DISCA_2011


C:\Users\phili\AppData\Local\Temp\ipykernel_24784\1446457672.py:192: DeprecationWarning: deprecated
  if hasattr(doc, "furniture") and doc.furniture and doc.furniture.children:


...

C:\Users\phili\AppData\Local\Temp\ipykernel_24784\1446457672.py:192: DeprecationWarning: deprecated
  if hasattr(doc, "furniture") and doc.furniture and doc.furniture.children:


  116 pages | 110 tables | 2570 blocks | 0 fn
  -> DVN_2007.json

DVN_2010


C:\Users\phili\AppData\Local\Temp\ipykernel_24784\1446457672.py:192: DeprecationWarning: deprecated
  if hasattr(doc, "furniture") and doc.furniture and doc.furniture.children:


  154 pages | 135 tables | 1789 blocks | 1 fn
  -> DVN_2010.json

OK: 27 | Errors: 0


{'success': 27, 'error': 0}